# 01 · Gaussian 表示新手教程

连续变量（CV / continuous variable）光量子里，**高斯态**用两样东西描述：

1. **均值向量** \(\bar r\)（displacement / 位移）
2. **协方差矩阵** \(V\)（covariance / 涨落与纠缠）

本教程只动 **`cvsim.gaussian`**：真空 → 挤压 → 位移 → 分束 → 损耗 → Homodyne。

配套笔记：`02-Gaussian表示原理.md`。

## 1. 这是啥 / 为啥用

- 激光近似真空 + 位移（相干态）是高斯的。
- 挤压光、分束器、多模线性光学：高斯门 **只改 \(V,\bar r\)**，不需要整本 Hilbert 空间。
- 成本：\(O(m^2)\) 量级，模数 \(m\) 可以比 Fock 大很多。

**一句话：** 你只关心「平均在哪 + 噪声椭圆长什么样」时，用 Gaussian。

## 2. 约定钉死（三表示共用）

| 项 | 值 |
|----|-----|
| \(\hbar\) | **1** |
| 正交序 | **xxpp**：\((x_1\ldots x_m, p_1\ldots p_m)\) |
| 真空 | \(V=I/2\)，\(\bar r=0\) |
| 纯单模高斯 | \(\det V = 1/4\) |
| 单模挤压 | \(\langle n\rangle = \sinh^2 r\) |
| 位移 | \(d_x=\sqrt{2}\mathrm{Re}\alpha\)，\(d_p=\sqrt{2}\mathrm{Im}\alpha\) |

In [ ]:
# 从仓库根启动 Jupyter 最稳；若在 tutorials/ 里打开，这里兜底加路径
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "cvsim").is_dir():
    ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
print("repo root:", ROOT)
print("numpy", np.__version__)

In [ ]:
from cvsim.gaussian import (
    GaussianState,
    beamsplitter,
    det_cov,
    displace,
    homodyne_condition,
    homodyne_mean,
    homodyne_sample,
    homodyne_var,
    loss,
    mean_photon,
    squeeze,
)

## 3. 最小闭环：真空 → 挤压

真空 \(V=I/2\)。沿 \(x\) 挤压（参数 \(r\)）后：

\[
V = \tfrac12\mathrm{diag}(e^{-2r}, e^{2r}),\quad
\det V = 1/4,\quad
\langle n\rangle = \sinh^2 r.
\]

In [ ]:
r = 0.8
vac = GaussianState.vacuum(1)
st = squeeze(vac, r=r, mode=0)

print("V =\n", st.V)
print("det V =", det_cov(st), "  expect 0.25")
print("<n>   =", mean_photon(st), "  expect", float(np.sinh(r) ** 2))
print("var x =", st.V[0, 0], "  expect", 0.5 * np.exp(-2 * r))
print("var p =", st.V[1, 1], "  expect", 0.5 * np.exp(+2 * r))

## 4. 数字检查：位移 + Homodyne

相干态 ≈ 真空位移。本约定 \(\langle x\rangle = \sqrt{2}\mathrm{Re}\alpha\)。

Homodyne（零差测量）测 \(x_\varphi = x\cos\varphi + p\sin\varphi\)；高斯边缘方差 \(u^\top V u\)。

API：`homodyne_mean(state, mode=0, phi=0.0)`。

In [ ]:
alpha = 1.2 + 0.0j
coh = displace(GaussianState.vacuum(1), alpha=alpha)
print("<n> ~ |alpha|^2 :", mean_photon(coh), "vs", abs(alpha) ** 2)
print("homodyne mean φ=0 :", homodyne_mean(coh, phi=0.0), "expect", np.sqrt(2) * alpha.real)
print("homodyne var  φ=0 :", homodyne_var(coh, phi=0.0), "expect ~0.5 (vacuum noise)")

## 5a. 再进一步：两模挤压 + 分束

模 0 先挤，再与真空模 1 做 50/50 BS。总光子数守恒（理想无损）。

In [ ]:
r = 0.6
two = GaussianState.vacuum(2)
two = squeeze(two, r=r, mode=0)
two = beamsplitter(two, 0, 1, theta=np.pi / 4)
print("total <n> :", mean_photon(two), "expect", float(np.sinh(r) ** 2))
print("det V     :", det_cov(two), "expect (1/4)^2 =", 0.0625)

## 5b. 损耗 loss 与条件 Homodyne

纯损耗：`loss(T)`，\(0\le T\le 1\)，环境真空。相干态 \(\langle n\rangle \to T|\alpha|^2\)。

`homodyne_condition(state, mode, phi, outcome)`：高斯 **Kalman 更新**——测完后测向方差 → 0，均值 → outcome。

In [ ]:
alpha, T = 1.5, 0.4
st = loss(displace(GaussianState.vacuum(1), alpha=alpha), T=T)
print("after loss <n>:", mean_photon(st), "expect", T * abs(alpha) ** 2)

# 条件测量：真空上「假装」测到 x=0.7
post = homodyne_condition(GaussianState.vacuum(1), mode=0, phi=0.0, outcome=0.7)
print("post mean x:", homodyne_mean(post, phi=0.0), "  var x:", homodyne_var(post, phi=0.0))

# 采样（随机抽一次结果；可设 seed）
rng = np.random.default_rng(0)
samples = [
    homodyne_sample(GaussianState.vacuum(1), phi=0.0, rng=rng) for _ in range(5)
]
print("5 vacuum samples:", samples)

## 6. 诚实边界 + 何时换表示

**Gaussian 适合**

- 线性光学 + 高斯通道（loss / 热 n̄）
- 大规模 GBS 的「态演化」侧（本包 **不做** Hafnian 采样）

**Gaussian 不适合 / 本包不做**

- 光子数分辨（PNRD）精确分布 → 用 **Fock**
- Cat / GKP 这种非高斯叠加 → 用 **Bosonic**
- Kerr 等非高斯门 → **Fock**（截断）

下一本：`02_fock_beginner.ipynb`。

## 自检（全绿才算过）

In [ ]:
r = 0.8
st = squeeze(GaussianState.vacuum(1), r=r)
assert abs(det_cov(st) - 0.25) < 1e-10
assert abs(mean_photon(st) - np.sinh(r) ** 2) < 1e-10
assert abs(homodyne_mean(displace(GaussianState.vacuum(1), 1.0), phi=0.0) - np.sqrt(2.0)) < 1e-10
post = homodyne_condition(GaussianState.vacuum(1), mode=0, phi=0.0, outcome=0.3)
assert abs(homodyne_mean(post, phi=0.0) - 0.3) < 1e-8
assert abs(homodyne_var(post, phi=0.0)) < 1e-8
print("T1 self-check OK")